##  1. 
###  a) distinct()  -  checks all columns in the DataFrame and returns only unique rows.\

### b) dropDuplicates()  -   allows you to pass specific subset columns. PySpark keeps the first encountered row for each unique combination of those columns and drops subsequent matches.\

###  Both to for find uniquw records

In [0]:
data  = [(1 , 'Bhupesh', 'M' , 5000) ,  (2 , 'Shivani', 'F' , 20000) ,(2 , 'Shivani', 'F' , 20000) , (3 , 'Rani' ,'F' , 10000)]
schema = ['id' , 'name' , 'gender' , 'salary']

df = spark.createDataFrame(data , schema)

df.show()

In [0]:
df.distinct().show()

In [0]:
df.dropDuplicates().show()

In [0]:
df.dropDuplicates(['gender']).show()       # when pass column name it  return only reords with unique gender


## 2. Union() & UnionAll()

**a) Are used to merge two or more dataframe of same schema or structure\
b) merges two dataframe and return the new dataFrame  regardless of duplicate records\
c) To remove duplicate use duplicate()**

In [0]:
data1 = [(1,'Bhupesh' ,'Male'), (2,'Shvani' ,'Female')]
schema1 =['id' , 'name' , 'gender']

data2 = [(3 , 'Bhupesh' ,'Male'), (4,'Rani' ,'Female')]
schema2 =['id' , 'name' , 'gender']

df = spark.createDataFrame(data1 , schema1)
df1 = spark.createDataFrame(data2 , schema2)

df.show()
df1.show()


new_df = df.unionAll(df1)
new_df.show()

final=new_df.distinct()
final.show()



## 3. GroupBY -  

**in PySpark collects rows with identical values in specified columns into groups, allowing you to run aggregate functions (like sum, average, count, min, or max) on each group.**

In [0]:
data = [
    (1 , 'Bhupesh', 1000, 'IT'),
    (2 , 'Ram', 6055, 'UI'),
    (3 , 'Bholu', 55433, 'UI'),
    (4 , 'Ayan', 50000, 'IT'),
    (5 , 'Laskhay', 20000, 'civil'),
    (6 , 'Geeta', 6000, 'UI'),

]
schema = ['id' , 'name' , 'salary' , 'department']

df = spark.createDataFrame(data , schema )
df.show()


df1=df.groupBy("department").count()
df1.show()

df2 = df.groupBy('department').min('salary')
df2.show()

df3=df.groupBy('department' ,'id').count()
df3.show()



### Aggregation funtion agg()  --- uses why help full 
### 
1. Running Multiple Aggregations at Once\
Direct methods only allow one metric on one column per operation. .agg() lets you calculate multiple metrics across different columns in a single pass over your dataset\

2. Renaming Columns Cleanly (.alias())\
Direct methods generate messy default column names like avg(salary) or sum(salary). With .agg(), you can clean up column names inline using .alias()

In [0]:
from pyspark.sql.functions import count,min, max

df4 = df.groupBy('department' ).agg(count('*').alias('CountOFEmployee'),\
                                    min('salary').alias('min(salary)'),\
                                    max('salary').alias('max(salary)'))
df4.show()



## 4. unionByName()   -  for merging two or more dataframes

### Key Differences union()  And unionByName() 
### 
### Union()
1 Matches strictly by position/index (1st col with 1st col, 2nd with 2nd).\
2 Mixes up data under wrong columns or causes data type mismatch errors.\
3 Fails if the number of columns differs between DataFrames.\

### unionByName()
1 Matches by column names, regardless of their order.\
2 Automatically reorders columns to match before merging.\
3 Can merge DataFrames with missing columns using allowMissingColumns=True.**

In [0]:
data1= [(1,'Bhupesh' ,23)]
schema1 = ['id' , 'name' , 'age']

data2 = [(2,'Shvani' ,4000)]
schema2 =['id' , 'name' , 'salary']

df1= spark.createDataFrame(data1 , schema1)
df2= spark.createDataFrame(data2 , schema2)

df1.show()
df2.show()
  
df_union = df1.union(df2)        # Altough column name are diffrent but the schema is same , that why is takes only first df1 column and leave  df2 unmatched column
df_union.show()

df_unionByName = df1.unionByName(df2 , allowMissingColumns=True)      # we using allowMissingColumns=True to get all column becuse column name is not similar 
df_unionByName.show()


## 5. Select() Function

**Used to select single , multiple column by index , all column from the list and the nested column from a DataFrame**

In [0]:
data = [(1,'Bhupesh' ,3000) ,(2,'Shivani' , 4000)]
schema = ['id' , 'name' , 'salary']

df = spark.createDataFrame(data , schema)
df.show(truncate=False)

df.select('name').show()

df.select('*').show()

## 6. join() 
## 
**we can combine columns from diffrenct dataFrmae  based on Condition , same as like SQL**

In [0]:
# df_emp (Employees)
emp_data = [(1, "Bhupesh", 10), (2, "Shivani", 20), (3, "Rani", 30)]
df_emp = spark.createDataFrame(emp_data, ["emp_id", "name", "dept_id"])

# df_dept (Departments)
dept_data = [(10, "IT"), (20, "HR"), (40, "Finance")]
df_dept = spark.createDataFrame(dept_data, ["dept_id", "dept_name"])



df_emp.join(df_dept, on="dept_id", how="inner").show()


### Important of JOIN()\
### 
### "left_semi  - Returns left rows that have a match in right DF (doesn't add right DF columns).\
### "left_anti  - Returns left rows that do not have a match in right DF.